In [1]:
import argparse
import logging
import os
import sys
import mercantile
from typing import List, cast
from osgeo import gdal
from pyproj import Transformer
from shapely.geometry import box
from src.stac.planetary_computer import query_planetary_computer_stac
from src.stac.stac_parameter_parser import parse_bbox, parse_time_window
from src.stac.stac_utils import get_bbox_and_footprint, order_stac, write_stac_meta

In [2]:
time = "2024-07-01/2024-08-27"
collection_id = "landsat-c2-l2"
x,y,z,p = 130371,86186,18,23
tile = mercantile.Tile(x, y, z)
tile_bbox = mercantile.bounds(tile)
left, bottom, right, top = (
    tile_bbox.west,
    tile_bbox.south,
    tile_bbox.east,
    tile_bbox.north,
)
bounds = ",".join(
            [
                str(i)
                for i in [
                    tile_bbox.west,
                    tile_bbox.south,
                    tile_bbox.east,
                    tile_bbox.north,
                ]
            ]
        )

# Get STAC items
items = query_planetary_computer_stac(time, bounds, collection_id)

C:\Users\Geo\miniconda3\envs\STACCLI\Lib\site-packages\pystac_client\item_search.py:903: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(


In [ ]:
ordered_features = order_stac(items)
file_path = "C:/Users/Geo/Desktop/output"
# open cogs
assets = "red,blue,green,nir08".split(",")
asset_bands = {assets[i]: i for i in range(len(assets))}
cog_urls = []

for feat in ordered_features:
    print("reading cogs.")
    for a in assets:
        cog_urls = cog_urls + [i["assets"][a]["href"] for i in [feat]]
    stac_id = feat["id"]

    outvrt = "/vsimem/stacked.vrt"  # /vsimem is special in-memory virtual "directory"

    polygon = box(*[left, bottom, right, top])
    print("building vrt.")
    outds = gdal.BuildVRT(outvrt, cog_urls, separate=True)

    # xres = outds.RasterXSize
    # yres = outds.RasterYSize
    print("translating tif.")
    outds = gdal.Translate(
        f"{file_path}/{stac_id}.tif",
        outds,
        # cog_urls, 
        # separate=True,
        # dstNodata=-1,
        # resampleAlg="bilinear",
        format="COG",
        # width=xres,
        # height=yres,
    ) # to patch for pred

    


reading cogs.
building vrt.
translating tif.
reading cogs.
building vrt.
translating tif.
reading cogs.
building vrt.
translating tif.


In [ ]:

zip_path = "C:/Users/Geo/Downloads/archive.zip"

from zipfile import ZipFile

with ZipFile(zip_path) as zf:
    for file in zf.namelist():
        if not file.startswith("8-Cloud_95-Cloud_Test_Metadata_Files/38-Cloud_95-Cloud_Test_Metadata_Files/"):
            print(file)
        
        # if not file.endswith('.png'): # optional filtering by filetype
        #     continue
        # with zf.open(file) as f:
        #     image = pygame.image.load(f, namehint=file)

In [ ]:
import rasterio

path = f'zip+file://{zip_path}!/38-Cloud_training/train_gt/gt_patch_218_10_by_2_LC08_L1TP_011002_20160620_20170323_01_T1.TIF'
dataset = rasterio.open(path)